# 02 — Exploratory Data Analysis

**Objective:** Perform comprehensive EDA on the synthetic maternal health dataset to understand feature distributions, correlations, and class characteristics.

## Steps
1. Load the synthetic dataset
2. Univariate analysis (distributions, summary stats)
3. Bivariate analysis (correlations, group comparisons by risk label)
4. Correlation heatmap
5. Class balance assessment
6. Key findings summary

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
%matplotlib inline

In [ ]:
# Load dataset
df = pd.read_csv('../data/synthetic/maternal_health_synthetic.csv')
print(f'Shape: {df.shape}')
df.info()

In [ ]:
# Univariate analysis — Continuous features
continuous_features = ['maternal_age', 'systolic_bp', 'diastolic_bp', 'hemoglobin',
                       'bmi', 'fasting_blood_glucose', 'weight_gain']

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.flatten()

for i, col in enumerate(continuous_features):
    ax = axes[i]
    df[col].hist(bins=40, ax=ax, color='steelblue', alpha=0.7, edgecolor='white')
    ax.axvline(df[col].mean(), color='red', linestyle='--', linewidth=1.5,
               label=f'Mean={df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='orange', linestyle=':', linewidth=1.5,
               label=f'Median={df[col].median():.1f}')
    ax.set_title(col.replace('_', ' ').title(), fontsize=11)
    ax.legend(fontsize=7)

# Discrete features
for i, col in enumerate(['parity', 'antenatal_visits'], start=len(continuous_features)):
    ax = axes[i]
    df[col].value_counts().sort_index().plot(kind='bar', ax=ax, color='teal', alpha=0.7)
    ax.set_title(col.replace('_', ' ').title(), fontsize=11)
    ax.set_xlabel('')

plt.suptitle('Univariate Distributions — All Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/eda_univariate.png', dpi=150, bbox_inches='tight')
plt.show()

# Categorical features
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df['socioeconomic_status'].value_counts().plot(kind='bar', ax=axes[0], color=['#e74c3c','#f39c12','#2ecc71'])
axes[0].set_title('Socioeconomic Status')
axes[0].set_xlabel('')

df['facility_type'].value_counts().plot(kind='bar', ax=axes[1], color='steelblue', alpha=0.7)
axes[1].set_title('Facility Type')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=20)

binary_cols = ['history_hypertension', 'gestational_diabetes', 'previous_cesarean', 'previous_preeclampsia']
prevalences = [df[c].mean() * 100 for c in binary_cols]
axes[2].barh([c.replace('_', ' ').title() for c in binary_cols], prevalences, color='coral')
axes[2].set_title('Binary Feature Prevalence (%)')
axes[2].set_xlabel('%')

plt.tight_layout()
plt.savefig('../reports/figures/eda_categorical.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Bivariate analysis — Box plots by risk label
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

continuous_features = ['maternal_age', 'systolic_bp', 'diastolic_bp', 'hemoglobin',
                       'bmi', 'fasting_blood_glucose', 'weight_gain', 'antenatal_visits']

colors = {0: '#2ecc71', 1: '#e74c3c'}
for i, col in enumerate(continuous_features):
    ax = axes[i]
    for label, color in colors.items():
        subset = df[df['risk_label'] == label][col]
        ax.hist(subset, bins=30, alpha=0.5, color=color,
                label=f'{"High" if label else "Low"} Risk', edgecolor='white')
    ax.set_title(col.replace('_', ' ').title(), fontsize=11)
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Risk Label', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/eda_bivariate_hist.png', dpi=150, bbox_inches='tight')
plt.show()

# Box plots
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, col in enumerate(continuous_features):
    sns.boxplot(data=df, x='risk_label', y=col, ax=axes[i],
                palette={0: '#2ecc71', 1: '#e74c3c'})
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=11)
    axes[i].set_xticklabels(['Low Risk', 'High Risk'])
    axes[i].set_xlabel('')

plt.suptitle('Box Plots by Risk Label', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/eda_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
numeric_df = df.select_dtypes(include=[np.number]).drop(columns=['patient_id'])

fig, ax = plt.subplots(figsize=(14, 11))
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix — Numeric Features', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/figures/eda_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Strongest correlations with risk_label
risk_corr = corr['risk_label'].drop('risk_label').abs().sort_values(ascending=False)
print("Feature correlations with risk_label (absolute):")
print(risk_corr.to_string())

In [ ]:
# Class balance assessment
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
counts = df['risk_label'].value_counts()
axes[0].bar(['Low Risk (0)', 'High Risk (1)'], counts.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='white')
axes[0].set_title('Risk Label Distribution (Counts)')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

# Proportion pie chart
axes[1].pie(counts.values, labels=['Low Risk', 'High Risk'],
            colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Risk Label Distribution (Proportion)')

plt.suptitle(f'Class Balance — High-Risk Prevalence: {df["risk_label"].mean():.1%}', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/figures/eda_class_balance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n## Key EDA Findings")
print(f"- Dataset: {len(df)} records, {df.shape[1]} features")
print(f"- High-risk prevalence: {df['risk_label'].mean():.1%} (imbalanced → will use SMOTE)")
print(f"- Mean maternal age: {df['maternal_age'].mean():.1f} ± {df['maternal_age'].std():.1f} years")
print(f"- Mean hemoglobin: {df['hemoglobin'].mean():.1f} ± {df['hemoglobin'].std():.1f} g/dL")
print(f"- Mean BMI: {df['bmi'].mean():.1f} ± {df['bmi'].std():.1f} kg/m²")
print(f"- History of hypertension prevalence: {df['history_hypertension'].mean():.1%}")
print(f"- GDM prevalence: {df['gestational_diabetes'].mean():.1%}")
print(f"- Previous CS prevalence: {df['previous_cesarean'].mean():.1%}")
print(f"- Strongest correlations with risk: {risk_corr.head(5).index.tolist()}")